# CMAPSS Data Exploration

Exploring the CMAPSS jet engine remaining useful life (RUL) dataset.
- Train datasets: FD001, FD002, FD003, FD004
- Features: Engine ID, cycle number, 21 sensor readings
- Target: Remaining useful life (cycles until failure)

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from src.dataloader import load_cmapss_data
from src.config import DATA_DIR, TRAIN_FILES

# Set visualization style
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

In [2]:
# Load training data FD001
print("Loading CMAPSS FD001 training dataset...")
train_fd001 = load_cmapss_data("train_FD001.txt")
test_fd001 = load_cmapss_data("test_FD001.txt")
rul_fd001 = pd.read_csv(DATA_DIR / "RUL_FD001.txt", header=None, names=["RUL"])

print(f"Train shape: {train_fd001.shape}")
print(f"Test shape: {test_fd001.shape}")
print(f"RUL shape: {rul_fd001.shape}")
print(f"\nFirst few rows of train data:\n{train_fd001.head()}")

Loading CMAPSS FD001 training dataset...
Train shape: (20631, 28)
Test shape: (13096, 28)
RUL shape: (100, 1)

First few rows of train data:
   0   1       2       3      4       5       6        7        8      9   \
0   1   1 -0.0007 -0.0004  100.0  518.67  641.82  1589.70  1400.60  14.62   
1   1   2  0.0019 -0.0003  100.0  518.67  642.15  1591.82  1403.14  14.62   
2   1   3 -0.0043  0.0003  100.0  518.67  642.35  1587.99  1404.20  14.62   
3   1   4  0.0007  0.0000  100.0  518.67  642.35  1582.79  1401.87  14.62   
4   1   5 -0.0019 -0.0002  100.0  518.67  642.37  1582.85  1406.22  14.62   

   ...       18      19    20   21    22     23     24       25  26  27  
0  ...  8138.62  8.4195  0.03  392  2388  100.0  39.06  23.4190 NaN NaN  
1  ...  8131.49  8.4318  0.03  392  2388  100.0  39.00  23.4236 NaN NaN  
2  ...  8133.23  8.4178  0.03  390  2388  100.0  38.95  23.3442 NaN NaN  
3  ...  8133.83  8.3682  0.03  392  2388  100.0  38.88  23.3739 NaN NaN  
4  ...  8133.80  8.4294  0

In [ ]:
# Define column names based on CMAPSS documentation
col_names = ["engine_id", "cycle", "op_setting_1", "op_setting_2", "op_setting_3"] + \
            [f"sensor_{i}" for i in range(1, 22)]

train_fd001.columns = col_names
test_fd001.columns = col_names

print("Data statistics:")
print(f"Number of engines: {train_fd001['engine_id'].nunique()}")
print(f"Cycles per engine: {train_fd001.groupby('engine_id')['cycle'].max().describe()}")
print(f"\nSensor statistics:\n{train_fd001[[f'sensor_{i}' for i in range(1, 6)]].describe()}")

In [ ]:
# Visualization: Sensor readings over cycles for first engine
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

engine_1 = train_fd001[train_fd001["engine_id"] == 1]

# Plot operating settings
axes[0].plot(engine_1["cycle"], engine_1["op_setting_1"], label="Op Setting 1")
axes[0].plot(engine_1["cycle"], engine_1["op_setting_2"], label="Op Setting 2")
axes[0].plot(engine_1["cycle"], engine_1["op_setting_3"], label="Op Setting 3")
axes[0].set_title("Operating Settings - Engine 1")
axes[0].set_xlabel("Cycle")
axes[0].legend()
axes[0].grid()

# Plot sensor readings
for i in range(1, 6):
    axes[1].plot(engine_1["cycle"], engine_1[f"sensor_{i}"], label=f"Sensor {i}", alpha=0.7)
axes[1].set_title("Sensor Readings (1-5) - Engine 1")
axes[1].set_xlabel("Cycle")
axes[1].legend()
axes[1].grid()

for i in range(6, 11):
    axes[2].plot(engine_1["cycle"], engine_1[f"sensor_{i}"], label=f"Sensor {i}", alpha=0.7)
axes[2].set_title("Sensor Readings (6-10) - Engine 1")
axes[2].set_xlabel("Cycle")
axes[2].legend()
axes[2].grid()

plt.tight_layout()
plt.show()